# Model Deployment\n
\n
This notebook demonstrates how to deploy the trained model for making predictions on new student data.\n
\n
We will load the saved model, scaler, and feature columns, then use them to make predictions.

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
np.random.seed(42)

## 1. Load Saved Model and Preprocessing Objects\n
\n
Load the trained model, scaler, and feature columns that were saved during training.

In [ ]:
# Load the trained model\n
with open('../models/risk_model_v1.pkl', 'rb') as f:
    model = pickle.load(f)

# Load the scaler\n
with open('../models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Load feature columns\n
with open('../models/feature_columns.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

print('Model and preprocessing objects loaded successfully!')
print(f'Number of features: {len(feature_cols)}')

## 2. Prepare New Data for Prediction\n
\n
Create a function to preprocess new student data in the same way as the training data.

In [ ]:
def preprocess_student_data(student_data):
    """
    Preprocess student data for prediction.
    
    Parameters:
    student_data (dict): Dictionary containing student information
    
    Returns:
    numpy array: Preprocessed features ready for model prediction
    """
    # Convert to DataFrame\n
    df = pd.DataFrame([student_data])
    
    # Ensure we have all required columns (fill missing with defaults)\n
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0  # Default value\n
    
    # Select only the features used in training\n
    df = df[feature_cols]
    
    # Scale the features\n
    df_scaled = scaler.transform(df)
    
    return df_scaled

## 3. Make Predictions\n
\n
Use the loaded model to make predictions on new student data.

In [ ]:
# Example: Predict risk for a new student\n
new_student = {
    'age': 20,
    'gender': 1,  # 1 for Male, 0 for Female (encoded)\n
    'avg_marks': 75.0,
    'std_marks': 10.0,
    'num_exams': 5,
    'avg_attendance': 80.0,
    'std_attendance': 15.0,
    'prop_present': 0.8,
    'prop_paid_fees': 0.8,
    'total_balance': 5000.0,
    'total_late_fine': 500.0,
    'assignments_completed': 10,
    'fee_status': 1,  # 1 for Paid, 0 for Overdue/Pending (encoded)\n
    'library_books_issued': 5,
    'complaints_count': 1,
    'behavior_score': 7,
    'course_risk': 0.15,
    'fee_pressure': 1,
    'academic_performance': 0.6,
    'engagement_score': 0.3
}

# Preprocess the data\n
processed_data = preprocess_student_data(new_student)

# Make prediction\n
risk_prediction = model.predict(processed_data)[0]
risk_probability = model.predict_proba(processed_data)[0][1]  # Probability of high risk\n

print(f'Risk Prediction: {"High Risk" if risk_prediction == 1 else "Low Risk"}')
print(f'Risk Probability: {risk_probability:.3f}')

# Interpretation\n
if risk_probability >= 0.7:
    print('\n⚠️  HIGH RISK: Immediate intervention recommended')
elif risk_probability >= 0.4:
    print('\n⚠️  MEDIUM RISK: Close monitoring advised')
else:
    print('\n✅ LOW RISK: Student appears to be on track')

## 4. Batch Prediction\n
\n
Make predictions for multiple students at once.

In [ ]:
# Example: Predict risk for multiple students\n
students_batch = [
    {  # Student 1: High risk\n
        'age': 21,
        'gender': 1,
        'avg_marks': 45.0,
        'std_marks': 15.0,
        'num_exams': 3,
        'avg_attendance': 40.0,
        'std_attendance': 20.0,
        'prop_present': 0.4,
        'prop_paid_fees': 0.3,
        'total_balance': 15000.0,
        'total_late_fine': 2000.0,
        'assignments_completed': 3,
        'fee_status': 0,
        'library_books_issued': 1,
        'complaints_count': 5,
        'behavior_score': 3,
        'course_risk': 0.25,
        'fee_pressure': 1,
        'academic_performance': 0.25,
        'engagement_score': -0.7
    },
    {  # Student 2: Low risk
        'age': 19,
        'gender': 0,
        'avg_marks': 85.0,
        'std_marks': 5.0,
        'num_exams': 6,
        'avg_attendance': 95.0,
        'std_attendance': 5.0,
        'prop_present': 0.95,
        'prop_paid_fees': 1.0,
        'total_balance': 0.0,
        'total_late_fine': 0.0,
        'assignments_completed': 15,
        'fee_status': 1,
        'library_books_issued': 10,
        'complaints_count': 0,
        'behavior_score': 9,
        'course_risk': 0.1,
        'fee_pressure': 0,
        'academic_performance': 0.9,
        'engagement_score': 1.0
    }
]

# Process each student
predictions = []
probabilities = []

for i, student in enumerate(students_batch):
    processed_data = preprocess_student_data(student)
    pred = model.predict(processed_data)[0]
    prob = model.predict_proba(processed_data)[0][1]
    predictions.append(pred)
    probabilities.append(prob)
    
    print(f'Student {i+1}: {"High Risk" if pred == 1 else "Low Risk"} (Probability: {prob:.3f})')

# Create results DataFrame\n
results_df = pd.DataFrame({
    'student_id': [f'STU_BATCH_{i+1:03d}' for i in range(len(students_batch))],
    'risk_prediction': predictions,
    'risk_probability': probabilities
})

print('\nBatch Prediction Results:')
display(results_df)

## 5. Save Prediction Results\n
\n
Save the prediction results for further analysis or integration with the web application.

In [ ]:
# Save batch predictions to CSV\n
results_df.to_csv('../datasets/batch_predictions.csv', index=False)
print('Batch predictions saved to ../datasets/batch_predictions.csv')

# Also save individual prediction for the first example\n
single_result = pd.DataFrame({
    'student_id': ['STU_SINGLE_001'],
    'risk_prediction': [risk_prediction],
    'risk_probability': [risk_probability]
})
single_result.to_csv('../datasets/single_prediction.csv', index=False)
print('Single prediction saved to ../datasets/single_prediction.csv')